# haRdQualizer — Goth character generator (SDXL, Colab T4)

Generates **original** semi-realistic goth-girl characters for the Alt/Gothic
visualizer, removes the background, and packages transparent PNGs that drop
straight into `assets/characters/`.

### Scope / ethics
Makes **original, fictional characters** in a goth *aesthetic* (makeup, horns,
spiked chokers, fishnet, platform boots, gold jewelry). It does **not** copy the
likeness of any real person — use reference photos only as a style moodboard.

### How to run
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Run the cells top to bottom.
3. **If cell 3 asserts Pillow >= 12:** Runtime -> Restart session, then run from
   cell 3 (skip cell 2). `rembg` upgrades Pillow and breaks `diffusers`; the pin
   + restart fixes it.
4. The last cell downloads `characters.zip`. Extract it into
   `haRdQualizer/assets/characters/` so you get `goth_a/body.png`, etc.

In [ ]:
# 1. Check the GPU (expect a Tesla T4, ~15GB).
!nvidia-smi

In [ ]:
# 2. Install dependencies (~2-3 min).
#
# IMPORTANT: rembg pulls in Pillow 12, which breaks Colab's diffusers
# ("ImportError: cannot import name '_Ink' from 'PIL._typing'"). We pin
# "pillow<12" in the SAME command. The cudf / cuml / numba / scikit-image
# warnings printed by pip are harmless (unused RAPIDS libs).
!pip install -q diffusers transformers accelerate safetensors rembg onnxruntime "pillow<12"

print("\n*** If pip changed Pillow: Runtime > Restart session, then run from cell 3.")

In [ ]:
# 3. Guard: confirm a diffusers-compatible Pillow before importing anything.
import PIL
major = int(PIL.__version__.split(".")[0])
print("Pillow", PIL.__version__)
assert major < 12, (
    "Pillow >= 12 is loaded and will break diffusers. "
    "Do Runtime > Restart session, then re-run from this cell (skip cell 2)."
)
print("Pillow OK")

In [ ]:
# 4. Load SDXL base ON THE GPU (not system RAM).
#
# Free Colab has only ~12.7 GB system RAM. enable_model_cpu_offload() keeps all
# weights in that RAM and overflows it -> "session crashed, used all available
# RAM". The T4 has 15 GB VRAM, which fits SDXL base in fp16, so we load straight
# onto CUDA and keep system RAM free.
import gc
import torch
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
    low_cpu_mem_usage=True,   # load shard-by-shard to keep the RAM peak small
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")        # weights live on the 15 GB T4, not system RAM
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

gc.collect(); torch.cuda.empty_cache()
print("pipeline ready on", pipe.device)
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

**If memory is still tight**
- *"Session crashed, used all RAM"* again -> Runtime -> **Restart session** to
  clear the previous crash's leftovers, then run cell 4 **once** (running it twice
  loads SDXL twice).
- *CUDA out of memory* -> lower `WIDTH, HEIGHT` to `768, 1024` in cell 5, or set
  `STEPS = 25`.
- Much lighter option: use a Stable Diffusion 1.5 checkpoint
  (`"runwayml/stable-diffusion-v1-5"`) with `StableDiffusionPipeline` — ~4x less
  memory, faster, slightly less detail.

In [ ]:
# 5. Configuration — edit the characters and style here.
#
# Keep a fixed `seed` per character so re-runs stay consistent. To push toward
# photoreal, edit STYLE (e.g. add 'photorealistic') — but original characters
# only, never a specific real person.

STYLE = (
    "{desc}, full body, standing straight, facing camera, arms hanging down at "
    "sides, symmetrical, highly detailed semi-realistic digital illustration, "
    "dramatic gothic makeup, pale skin, dark lips, sharp eyeliner, clean cel "
    "shading with soft rendering, full figure visible head to feet, centered, "
    "plain flat light-grey studio background, even lighting"
)

NEG = (
    "photograph of a real person, celebrity, specific identity, multiple people, "
    "cropped, out of frame, cut off legs, extra limbs, extra fingers, bad hands, "
    "deformed, blurry, lowres, watermark, signature, text, logo, nsfw, nude, "
    "oversexualized, busy background, props"
)

CHARACTERS = [
    {
        "name": "goth_a",
        "seed": 12345,
        "desc": "young goth woman, long straight black hair with blunt bangs, "
                "small red devil horns, spiked black leather choker, black mesh "
                "long-sleeve top, black pleated mini skirt, fishnet tights, "
                "chunky platform boots",
    },
    {
        "name": "goth_b",
        "seed": 5544,
        "desc": "young goth woman, long wavy dark hair, gold forehead chain, "
                "layered gold necklaces, black corset dress with gold accents, "
                "lace gloves, fishnet tights, platform boots",
    },
    {
        "name": "goth_c",
        "seed": 909090,
        "desc": "young goth woman, shoulder-length black hair with dark green "
                "streaks, studded choker, oversized band t-shirt, black tutu "
                "skirt, ripped fishnets, platform boots",
    },
]

# T4-friendly generation settings.
STEPS = 30
GUIDANCE = 6.5
WIDTH, HEIGHT = 832, 1216   # SDXL portrait, good for full-body framing

In [ ]:
# 6. Generate one clean full-body image per character.
from pathlib import Path

RAW_DIR = Path("/content/raw")
RAW_DIR.mkdir(exist_ok=True)
raw_images = {}

def generate(prompt, seed):
    g = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt=prompt, negative_prompt=NEG,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE,
        width=WIDTH, height=HEIGHT, generator=g,
    ).images[0]
    torch.cuda.empty_cache()   # release transient VRAM between characters
    return img

for c in CHARACTERS:
    prompt = STYLE.format(desc=c["desc"])
    print(f"generating {c['name']} (seed {c['seed']}) ...")
    img = generate(prompt, c["seed"])
    img.save(RAW_DIR / f"{c['name']}.png")
    raw_images[c["name"]] = img
print("done")

In [ ]:
# 7. Preview the raw renders. Re-run cells 5-6 with new seeds for any you dislike.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(raw_images), figsize=(4 * len(raw_images), 7))
if len(raw_images) == 1:
    axes = [axes]
for ax, (name, img) in zip(axes, raw_images.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.show()

In [ ]:
# 8. Remove the background -> transparent body.png per character.
from rembg import remove

OUT_DIR = Path("/content/characters")
OUT_DIR.mkdir(exist_ok=True)

for c in CHARACTERS:
    cut = remove(raw_images[c["name"]])  # RGBA with transparent background
    d = OUT_DIR / c["name"]
    d.mkdir(parents=True, exist_ok=True)
    cut.save(d / "body.png")
    print("saved", d / "body.png")

## Optional: pose variants (hands_up / dance)

The rig already animates the single `body.png` with sway / breathe / hop. For
distinct pose art, generate variants below with img2img. They save as
`hands_up.png` / `dance.png`; the rig swaps to them when that pose is active.

In [ ]:
# 9. (Optional) Pose variants via img2img. Skip if you only want body.png.
#
# Reuses the base pipeline's components (weights already on the GPU), so it adds
# no extra VRAM/RAM. Do NOT call enable_model_cpu_offload here.
from diffusers import StableDiffusionXLImg2ImgPipeline

img2img = StableDiffusionXLImg2ImgPipeline(**pipe.components)

POSE_PROMPTS = {
    "hands_up": "both arms raised straight overhead, hands up, cheering",
    "dance": "dancing pose, arms out to the sides, dynamic",
}
STRENGTH = 0.5  # lower = closer to body.png, higher = more pose change

for c in CHARACTERS:
    base = raw_images[c["name"]]
    for pose, extra in POSE_PROMPTS.items():
        prompt = STYLE.format(desc=c["desc"]) + ", " + extra
        g = torch.Generator(device="cuda").manual_seed(c["seed"])
        out = img2img(
            prompt=prompt, negative_prompt=NEG, image=base,
            strength=STRENGTH, num_inference_steps=STEPS,
            guidance_scale=GUIDANCE, generator=g,
        ).images[0]
        torch.cuda.empty_cache()
        cut = remove(out)
        (OUT_DIR / c["name"]).mkdir(parents=True, exist_ok=True)
        cut.save(OUT_DIR / c["name"] / f"{pose}.png")
        print("saved", c["name"], pose)

In [ ]:
# 10. Package everything and download.
import shutil
from google.colab import files

shutil.make_archive("/content/characters", "zip", "/content/characters")
print("Extract characters.zip into haRdQualizer/assets/characters/")
files.download("/content/characters.zip")